# Masroofy SMS — all results in one place

Two parts, every number shown as a table and a figure **inside this notebook** (nothing is written to disk):

| part | what | where the numbers come from |
|---|---|---|
| **1. Old implementation (v1)** | `sms_eval_final/sms_model.py` — one sklearn model that both detects and extracts transactions | **re-computed here** with v1's own `evaluate.py` code (its four splits), so confusion matrices are available |
| **2. New benchmark (sms_v2)** | Stage-1 classifiers (TF-IDF LR / SVM / NB, MARBERT, AraBERT, XLM-R, v1 gate) → Stage-2 extraction → end to end | read from the benchmark's results folder (`sms_v2/outputs/` or a copy of the Colab/Drive one) |
| **3. Old vs new** | how v1's own numbers relate to the benchmark's | both |

Run top to bottom (**Kernel → Restart & Run All**). Part 1 retrains v1 30 times (~5–10 min on CPU); set
`RUN_V1 = False` below to skip it. Transformer rows appear in part 2 automatically once their results exist.

## 0. Settings

In [1]:
import os
# Results of the sms_v2 benchmark. Default: the local run. For the transformer results, download the Drive
# folder MyDrive/masroofy_sms_v2/outputs and point this at it, e.g. r'C:\Users\me\Downloads\outputs'.
BENCH_OUT = os.environ.get('SMS_V2_OUTPUT') or None     # None = <repo>/sms_v2/outputs
SPLIT = 'template'           # the benchmark's primary split
RUN_V1 = True                # part 1: re-run v1's evaluation (~5-10 min); False = skip part 1
N_JOBS = 2                   # v1 folds trained in parallel; each needs ~0.3-0.5 GB RAM
N_BOOT = 1000                # bootstrap resamples (by template) for the 95 % intervals in part 2

In [2]:
import csv, glob, json, sys, time, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60, 'display.width', 200, 'display.precision', 3)

# repo root = the folder holding sms_eval_final/ and sms_v2/, found from wherever the notebook runs
REPO = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO, 'sms_eval_final')):
    parent = os.path.dirname(REPO)
    if parent == REPO:
        raise FileNotFoundError('run this notebook from inside the Masroofy repo (results/)')
    REPO = parent
V1_DIR = os.path.join(REPO, 'sms_eval_final')
BENCH = BENCH_OUT or os.path.join(REPO, 'sms_v2', 'outputs')
RES = os.path.join(BENCH, 'results', SPLIT)
print('repo     ', REPO)
print('v1       ', V1_DIR)
print('benchmark', RES, '(found)' if os.path.isdir(RES) else '(NOT FOUND: part 2 will be empty)')

# ---- one visual system for every figure (same palette as the benchmark's own charts)
TEXT, TEXT2, GRID = '#191919', '#666666', '#e5e5e5'
MODELS = ['tfidf_lr', 'tfidf_svm', 'tfidf_nb', 'marbert', 'arabert', 'xlm_roberta', 'v1_gate']
MODEL_COLOR = {'tfidf_lr': '#257de0', 'tfidf_svm': '#f47230', 'tfidf_nb': '#00b894', 'marbert': '#f2a600',
               'arabert': '#ee7fac', 'xlm_roberta': '#008a17', 'v1_gate': '#6a4ce2', 'oracle': '#666666'}
NAME = {'tfidf_lr': 'TF-IDF + LR', 'tfidf_svm': 'TF-IDF + SVM', 'tfidf_nb': 'TF-IDF + NB',
        'marbert': 'MARBERT', 'arabert': 'AraBERT', 'xlm_roberta': 'XLM-R', 'v1_gate': 'v1 gate',
        'oracle': 'oracle (perfect Stage 1)'}
EXP_NAME = {'binary': 'A: binary', 'multiclass': 'B: 6-class'}
V1_COLOR = MODEL_COLOR['v1_gate']        # v1 keeps its colour in every figure

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.titlesize': 10, 'axes.titlelocation': 'left',
                     'axes.titlecolor': TEXT, 'axes.labelcolor': TEXT2, 'axes.edgecolor': GRID,
                     'xtick.color': TEXT2, 'ytick.color': TEXT2, 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False, 'figure.facecolor': 'white'})


def style(ax, title=None, ylabel=None, grid='y'):
    if title: ax.set_title(title)
    if ylabel: ax.set_ylabel(ylabel)
    ax.tick_params(which='both', length=0)
    ax.spines['left'].set_visible(False)
    if grid:
        getattr(ax, f'{grid}axis').grid(True, color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)


def bars(ax, labels, values, color, fmt='{:.2f}', ylim=(0, 1.1), err=None):
    x = np.arange(len(labels))
    v = [np.nan if a is None else a for a in values]
    ax.bar(x, np.nan_to_num(v), 0.62, color=color, edgecolor='white', linewidth=1)
    if err is not None:
        lo, hi = np.array(err).T
        ax.errorbar(x, v, yerr=[np.array(v) - lo, hi - np.array(v)], fmt='none', ecolor=TEXT2, lw=0.8, capsize=2)
    for xi, vi in zip(x, v):
        if not np.isnan(vi):
            ax.text(xi, vi, ' ' + fmt.format(vi), ha='center', va='bottom', fontsize=7, color=TEXT2)
    ax.set_xticks(x, labels)
    if ylim: ax.set_ylim(*ylim)


def grouped(ax, groups, series, value, fmt='{:.2f}', ylim=(0, 1.12), err=None, label=lambda s: NAME.get(s, s)):
    # groups on x, one bar per series (model) in its fixed colour; value(g, s) -> number or None
    n = max(len(series), 1)
    w = 0.8 / n
    x = np.arange(len(groups))
    for i, s in enumerate(series):
        pos = x - 0.4 + w * (i + 0.5)
        vals = [value(g, s) for g in groups]
        h = [0 if v is None else v for v in vals]
        ax.bar(pos, h, w * 0.92, color=MODEL_COLOR.get(s, '#9c9c9c'), label=label(s), edgecolor='white', lw=1)
        for j, (p, v) in enumerate(zip(pos, vals)):
            if v is None:
                continue
            if err is not None and err(groups[j], s) is not None:
                lo, hi = err(groups[j], s)
                ax.errorbar(p, v, yerr=[[v - lo], [hi - v]], fmt='none', ecolor=TEXT2, lw=0.7, capsize=1.5)
            ax.text(p, v, fmt.format(v), ha='center', va='bottom', fontsize=6.3, color=TEXT2, zorder=5,
                    bbox=dict(boxstyle='square,pad=0.08', fc='white', ec='none', alpha=0.85))
    ax.set_xticks(x, groups)
    if ylim: ax.set_ylim(*ylim)


def heat(ax, M, rows, cols, fmt='{:.2f}', vmin=0, vmax=1, cmap='Blues'):
    M = np.array(M, dtype=float)
    ax.imshow(np.nan_to_num(M, nan=0), cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isnan(M[i, j]):
                ax.text(j, i, '–', ha='center', va='center', fontsize=7, color=TEXT2)
            else:
                dark = (M[i, j] - vmin) / max(vmax - vmin, 1e-9) > 0.6
                ax.text(j, i, fmt.format(M[i, j]), ha='center', va='center', fontsize=7,
                        color='white' if dark else TEXT)
    ax.set_xticks(range(len(cols)), cols, rotation=35, ha='right')
    ax.set_yticks(range(len(rows)), rows)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)


def confusion(ax, M, labels, title):
    # counts printed, colour = share of the gold row (so small classes are readable)
    M = np.array(M)
    norm = M / np.maximum(M.sum(1, keepdims=True), 1)
    ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(M[i, j]), ha='center', va='center', fontsize=7.5,
                    color='white' if norm[i, j] > 0.55 else TEXT)
    ax.set_xticks(range(len(labels)), labels, rotation=40, ha='right', fontsize=7)
    ax.set_yticks(range(len(labels)), labels, fontsize=7)
    ax.set_xlabel('predicted', fontsize=8)
    ax.set_ylabel('gold', fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)


def legend_below(fig, ax, ncol=4):
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=ncol, fontsize=8, bbox_to_anchor=(0.5, -0.04))

repo      C:\Users\mirol\Documents\Github\Masroofy
v1        C:\Users\mirol\Documents\Github\Masroofy\sms_eval_final
benchmark C:\Users\mirol\Documents\Github\Masroofy\sms_v2\outputs\results\template (found)


---
# Part 1 — Old implementation (v1, `sms_eval_final`)

v1 is a single sklearn model (`SmsTxnModel`): a type classifier with a `none` class decides *transaction or
not*, then number-role, merchant-tagging and date/currency rules fill the record. Its evaluation
(`evaluate.py`) uses four splits:

| split | test messages are … | how optimistic |
|---|---|---|
| `random` | 5-fold random | **very** — sibling messages from the same template sit in training |
| `unseen_wording` | 5-fold, grouped by template: wordings never seen in training | honest; closest to the benchmark's `template` split |
| `unseen_sender` | leave one sender out (19 folds): a whole bank's format never seen | hardest |
| `real_formats` | train on fictional senders only, test on the 1 200 real-format messages | closest to production |

The cells below run **v1's own code** (`evaluate._fold`, `evaluate.score`, the same fold definitions), so
the numbers should equal `sms_eval_final/eval_2200/eval_results.json` (checked below). v1's metric
definitions: transaction precision/recall over all messages; field accuracy **only on messages both gold and
v1 call transactions**; `full_record_exact` = every field right, over all gold transactions.

In [3]:
if RUN_V1:
    sys.path.insert(0, V1_DIR)
    import evaluate as V1E
    from sklearn.model_selection import KFold, GroupKFold, LeaveOneGroupOut

    data = [json.loads(l) for l in open(os.path.join(V1_DIR, 'sms_dataset_2200.jsonl'), encoding='utf-8') if l.strip()]
    man = {int(r['id']): r for r in csv.DictReader(open(os.path.join(V1_DIR, 'manifest_2200.csv'), encoding='utf-8-sig'))}
    prov = lambda r: man[r['id']].get('provenance') or 'fictional'
    idx = list(range(len(data)))
    tmpl = [man[r['id']]['template_id'] for r in data]
    send = [r['sender'] for r in data]

    def v1_folds(name):        # identical to evaluate.py's __main__
        if name == 'random': return list(KFold(5, shuffle=True, random_state=0).split(idx))
        if name == 'unseen_wording': return list(GroupKFold(5).split(idx, groups=tmpl))
        if name == 'unseen_sender': return list(LeaveOneGroupOut().split(idx, groups=send))
        if name == 'real_formats':
            tr = [i for i in idx if prov(data[i]) == 'fictional']
            te = [i for i in idx if prov(data[i]) != 'fictional']
            return [(tr, te)]

    V1_SPLITS = ['random', 'unseen_wording', 'unseen_sender', 'real_formats']
    V1_PAIRS = {}
    for name in V1_SPLITS:
        t0 = time.time()
        folds = v1_folds(name)
        V1_PAIRS[name] = V1E.run(data, folds, N_JOBS)
        print(f'{name:15s} {len(folds):2d} folds, {len(V1_PAIRS[name]):4d} test predictions, {time.time() - t0:5.0f} s')
else:
    print('RUN_V1 = False: part 1 skipped')

random           5 folds, 2200 test predictions,    62 s


unseen_wording   5 folds, 2200 test predictions,    33 s


MemoryError: Unable to allocate 909. KiB for an array with shape (3, 38791) and data type float64

### 1.1 Headline metrics per split

In [ ]:
if RUN_V1:
    V1_RES = {n: V1E.score(p)[0] for n, p in V1_PAIRS.items()}
    v1_tab = pd.DataFrame(V1_RES).T
    display(v1_tab.style.format(lambda v: '–' if v is None or (isinstance(v, float) and np.isnan(v)) else
                                (f'{v:.0f}' if v > 1 else f'{v:.3f}')).set_caption('v1, recomputed now'))

    saved_path = os.path.join(V1_DIR, 'eval_2200', 'eval_results.json')
    if os.path.exists(saved_path):
        saved = json.load(open(saved_path, encoding='utf-8'))
        diffs = {n: max(abs((V1_RES[n][k] or 0) - (saved[n][k] or 0)) for k in V1_RES[n] if k in saved.get(n, {}))
                 for n in V1_RES if n in saved}
        print('largest difference from the saved eval_2200/eval_results.json, per split:',
              {k: round(v, 4) for k, v in diffs.items()})
        print('(0 = identical; small differences can come from a different scikit-learn version)')

**About the check above.** On this machine (scikit-learn 1.7.2) `random`, `unseen_sender` and `real_formats`
reproduce the saved results exactly. `unseen_wording` differs by up to ~0.03 (e.g. transaction F1 0.869 vs 0.874,
full record 0.443 vs 0.422). It is the only split built with `GroupKFold`, so the likely cause is that this
scikit-learn version assigns the templates to its 5 folds differently from the version that produced the saved
file (the saved v1 model was pickled with scikit-learn 1.9.1) — same method, different folds. Not verified by
re-running under 1.9.1.

In [ ]:
if RUN_V1:
    mets = [('tx_precision', 'transaction precision'), ('tx_recall', 'transaction recall'),
            ('tx_f1', 'transaction F1'), ('non_tx_rejected', 'non-transactions rejected'),
            ('full_record_exact', 'full record exactly right')]
    fig, axes = plt.subplots(1, len(mets), figsize=(15, 3.1), sharey=True)
    labels = [s.replace('_', '\n') for s in V1_SPLITS]
    for ax, (k, title) in zip(axes, mets):
        bars(ax, labels, [V1_RES[s][k] for s in V1_SPLITS], V1_COLOR)
        style(ax, title)
    axes[0].set_ylabel('score')
    fig.suptitle('v1 — detection and record quality by split', x=0.01, ha='left', fontsize=11)
    fig.tight_layout()
    plt.show()

### 1.2 Field accuracy (on messages both gold and v1 call transactions)

In [ ]:
if RUN_V1:
    F = V1E.FIELDS
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    heat(ax, [[V1_RES[s][f] for s in V1_SPLITS] for f in F], F, V1_SPLITS)
    ax.set_title('v1 — field accuracy by split')
    plt.show()

### 1.3 Confusion matrices

**Transaction vs not** (v1's Stage-1 decision), then **transaction type** — gold type against predicted
type over *all* messages, with `not a transaction` as its own row/column, so a missed transaction shows up in
the last column and a false detection in the last row.

In [ ]:
if RUN_V1:
    def tx_cm(pairs):
        M = np.zeros((2, 2), int)
        for r, p in pairs:
            M[0 if r['expected']['is_transaction'] else 1, 0 if p['is_transaction'] else 1] += 1
        return M

    fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
    for ax, s in zip(axes, V1_SPLITS):
        confusion(ax, tx_cm(V1_PAIRS[s]), ['transaction', 'not'], f'{s}  (n={len(V1_PAIRS[s])})')
    fig.tight_layout()
    plt.show()

In [ ]:
if RUN_V1:
    NOT = 'not a transaction'
    TYPES = ['debit', 'credit', 'transfer_out', 'transfer_in', 'atm_withdrawal', 'bill_payment', 'other', NOT]

    def type_of(rec):
        return (rec.get('type') or 'other') if rec['is_transaction'] else NOT

    fig, axes = plt.subplots(2, 2, figsize=(12, 10.5))
    for ax, s in zip(axes.ravel(), V1_SPLITS):
        M = np.zeros((len(TYPES), len(TYPES)), int)
        for r, p in V1_PAIRS[s]:
            M[TYPES.index(type_of(r['expected'])), TYPES.index(type_of(p))] += 1
        confusion(ax, M, TYPES, f'v1 type — {s}')
    fig.tight_layout()
    plt.show()

### 1.4 By sender (bank / wallet)

In [ ]:
if RUN_V1:
    by_sender = {s: V1E.breakdown(V1_PAIRS[s], lambda r: r['sender']) for s in V1_SPLITS}
    senders = sorted({k for d in by_sender.values() for k in d})
    fig, axes = plt.subplots(1, 3, figsize=(15, 6.5), sharey=True)
    for ax, (k, title) in zip(axes, [('tx_f1', 'transaction F1'), ('non_tx_rejected', 'non-transactions rejected'),
                                     ('full_record_exact', 'full record exact')]):
        M = [[by_sender[s].get(snd, {}).get(k, np.nan) if by_sender[s].get(snd, {}).get(k) is not None else np.nan
              for s in V1_SPLITS] for snd in senders]
        heat(ax, M, senders, V1_SPLITS)
        ax.set_title(f'v1 — {title}')
    fig.tight_layout()
    plt.show()
    print('– = the sender has no messages of that kind in that split (e.g. no non-transactions to reject)')

### 1.5 By provenance (fictional senders vs real bank formats)

In [ ]:
if RUN_V1:
    rows = []
    for s in V1_SPLITS:
        for pv, r in V1E.breakdown(V1_PAIRS[s], prov).items():
            rows.append({'split': s, 'provenance': pv, 'n': r['n'], 'tx_f1': r['tx_f1'],
                         'non_tx_rejected': r['non_tx_rejected'], 'full_record_exact': r['full_record_exact'],
                         'amount': r['amount'], 'merchant': r['merchant'], 'datetime': r['datetime']})
    display(pd.DataFrame(rows).set_index(['split', 'provenance']))

### 1.6 Examples of v1's detection errors (honest split `unseen_wording`)

In [ ]:
if RUN_V1:
    errs = [(r, p) for r, p in V1_PAIRS['unseen_wording'] if r['expected']['is_transaction'] != p['is_transaction']]
    ex = pd.DataFrame([{'error': 'missed transaction' if r['expected']['is_transaction'] else 'false detection',
                        'sender': r['sender'], 'template': man[r['id']]['template_id'], 'sms': r['sms'][:110]}
                       for r, p in errs])
    print(len(errs), 'detection errors; by kind and template:')
    display(ex.groupby(['error', 'template']).size().rename('n').sort_values(ascending=False).head(15).to_frame())
    display(ex.groupby('error').head(4))

---
# Part 2 — New benchmark (`sms_v2`), split `template`

Every model is trained on the **same** train split, tuned on validation, and scored **once** on a test set of
message templates never seen in training (no near-duplicates across splits). Experiment **A** is binary
(transaction / not); **B** is 6-class (transaction, balance inquiry, OTP, promotional, failed transaction,
other) — only `transaction` is sent to extraction in both. Rows for models that have not been run yet
(e.g. the transformers before Colab) are simply absent.

In [ ]:
def load_runs():
    rows, preds = [], {}
    for exp in ('binary', 'multiclass'):
        for d in sorted(glob.glob(os.path.join(RES, exp, '*'))):
            if not os.path.exists(os.path.join(d, 'DONE.json')):
                continue
            m = json.load(open(os.path.join(d, 'metrics.json'), encoding='utf-8'))
            t, rt = m['test'], m['test']['routing']
            rows.append({
                'experiment': exp, 'model': m['model'], 'family': m['family'],
                'accuracy': t['accuracy'], 'macro_precision': t['macro_precision'], 'macro_recall': t['macro_recall'],
                'macro_f1': t['macro_f1'], 'weighted_f1': t['weighted_f1'],
                'tx_precision': rt['transaction_precision'], 'tx_recall': rt['transaction_recall'],
                'tx_f1': rt['transaction_f1'], 'non_tx_precision': rt['non_transaction_precision'],
                'non_tx_recall': rt['non_transaction_recall'], 'missed_tx': rt['missed_transactions'],
                'false_tx': rt['false_transactions'], 'val_macro_f1': m['val']['macro_f1'],
                'train_s': m['timing']['train_time_s'], 'ms_per_sms_batch': m['timing']['batch_ms_per_sms'],
                'ms_per_sms_single': m['timing']['single_ms_per_sms_median'], 'size_mb': m['model_size_mb'],
                'params_m': (m['n_parameters'] or 0) / 1e6 or None,
                'gpu_train_mb': (m['train'] or {}).get('peak_gpu_mem_train_mb'),
                'device': (m['train'] or {}).get('device', 'cpu'), 'meta': m})
            preds[(exp, m['model'])] = pd.read_csv(os.path.join(d, 'predictions.csv'), encoding='utf-8-sig')
    df = pd.DataFrame(rows)
    if len(df):
        df['_order'] = df.model.map({m: i for i, m in enumerate(MODELS)})
        df = df.sort_values(['experiment', '_order']).drop(columns='_order').reset_index(drop=True)
    return df, preds


CLS, PREDS = load_runs()
have = {(r.experiment, r.model) for r in CLS.itertuples()}
status = pd.DataFrame([[('done' if (e, m) in have else ('n/a (binary only)' if m == 'v1_gate' and e == 'multiclass'
                                                        else 'not run yet')) for e in ('binary', 'multiclass')]
                       for m in MODELS], index=[NAME[m] for m in MODELS], columns=['A: binary', 'B: 6-class'])
display(status)
EXPS = [e for e in ('binary', 'multiclass') if e in set(CLS.experiment)] if len(CLS) else []
RUN_MODELS = [m for m in MODELS if m in set(CLS.model)] if len(CLS) else []

### 2.1 Classification on the test set

`tx_*` = the decision that matters for the pipeline: is this SMS sent to extraction? `missed_tx` are real
transactions lost; `false_tx` are non-transactions that would become false records. `val_macro_f1` is the
validation score used for model selection — compare it with the test score to see how much the unseen
templates cost.

In [ ]:
if len(CLS):
    cols = ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'tx_precision', 'tx_recall',
            'tx_f1', 'non_tx_precision', 'non_tx_recall', 'missed_tx', 'false_tx', 'val_macro_f1']
    t = CLS.set_index(['experiment', 'model'])[cols]
    display(t.style.format('{:.3f}', subset=[c for c in cols if c not in ('missed_tx', 'false_tx')])
             .background_gradient('Blues', subset=['macro_f1', 'tx_recall', 'tx_precision'], vmin=0, vmax=1)
             .background_gradient('Oranges', subset=['missed_tx', 'false_tx']))

In [ ]:
# 95 % intervals: resample the test set by TEMPLATE (errors come in whole templates, so this is the honest unit)
TPL = {int(r['id']): r['template_id'] for r in csv.DictReader(open(os.path.join(V1_DIR, 'manifest_2200.csv'), encoding='utf-8-sig'))}
TX = 'transaction'


def boot(pred, n=N_BOOT, seed=0):
    classes = sorted(set(pred.gold) | set(pred.pred))
    ci = {c: i for i, c in enumerate(classes)}
    g = pred.gold.map(ci).values
    p = pred.pred.map(ci).values
    groups = pred.id.map(TPL).values
    uniq = np.unique(groups)
    members = [np.where(groups == u)[0] for u in uniq]
    k, tx = len(classes), ci.get(TX)
    rng = np.random.default_rng(seed)
    out = defaultdict(list)
    for _ in range(n):
        ix = np.concatenate([members[j] for j in rng.integers(0, len(uniq), len(uniq))])
        M = np.bincount(g[ix] * k + p[ix], minlength=k * k).reshape(k, k)
        tp, sup, prd = np.diag(M), M.sum(1), M.sum(0)
        prec = np.divide(tp, prd, out=np.zeros(k), where=prd > 0)
        rec = np.divide(tp, sup, out=np.zeros(k), where=sup > 0)
        f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(k), where=prec + rec > 0)
        out['macro_f1'].append(f1[sup > 0].mean())
        out['tx_recall'].append(rec[tx])
        out['tx_precision'].append(prec[tx])
    return {m: tuple(np.percentile(v, [2.5, 97.5])) for m, v in out.items()}


CI = {key: boot(df) for key, df in PREDS.items()}
print(f'{N_BOOT} template-bootstrap resamples per model; test set has',
      PREDS[next(iter(PREDS))].id.map(TPL).nunique() if PREDS else 0, 'templates')

In [ ]:
if len(CLS):
    val = lambda k: (lambda g, s: CLS.query('experiment == @g and model == @s')[k].iloc[0]
                     if ((g, s) in have) else None)
    eg = [EXP_NAME[e] for e in EXPS]
    inv = {v: k for k, v in EXP_NAME.items()}
    panels = [('macro_f1', 'macro F1'), ('tx_recall', 'transaction recall (missed = 1 − this)'),
              ('tx_precision', 'transaction precision (false records)'), ('non_tx_precision', 'non-transaction precision')]
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.6), sharey=True)
    for ax, (k, title) in zip(axes, panels):
        grouped(ax, eg, RUN_MODELS, lambda g, s, k=k: val(k)(inv[g], s),
                err=(lambda g, s, k=k: CI.get((inv[g], s), {}).get(k)) if k in ('macro_f1', 'tx_recall', 'tx_precision') else None)
        style(ax, title)
    axes[0].set_ylabel('test score')
    legend_below(fig, axes[0], ncol=len(RUN_MODELS))
    fig.suptitle('Stage 1 on unseen templates — whiskers: 95 % template-bootstrap interval', x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()

### 2.2 Per-class F1

In [ ]:
for exp in EXPS:
    sub = CLS[CLS.experiment == exp]
    classes = sub.iloc[0]['meta']['classes']
    M = [[r['meta']['test']['per_class'][c]['f1'] for c in classes] for _, r in sub.iterrows()]
    support = sub.iloc[0]['meta']['test']['per_class']
    fig, ax = plt.subplots(figsize=(1.6 + 1.25 * len(classes), 0.9 + 0.45 * len(sub)))
    heat(ax, M, [NAME[m] for m in sub.model], [f"{c}\n(n={support[c]['support']})" for c in classes])
    ax.set_title(f'{EXP_NAME[exp]} — F1 per class (test)')
    plt.show()

### 2.3 Confusion matrices

In [ ]:
for exp in EXPS:
    sub = CLS[CLS.experiment == exp].reset_index(drop=True)
    n = len(sub)
    ncol = min(n, 4)
    nrow = int(np.ceil(n / ncol))
    k = len(sub.iloc[0]['meta']['classes'])
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * (1.2 + 0.62 * k), nrow * (1.1 + 0.6 * k)), squeeze=False)
    for ax in axes.ravel()[n:]:
        ax.axis('off')
    for ax, (_, r) in zip(axes.ravel(), sub.iterrows()):
        cm = r['meta']['test']['confusion_matrix']
        short = [c.replace('_transaction', '').replace('_inquiry', '') for c in cm['labels']]
        confusion(ax, cm['matrix'], short, NAME[r.model])
    fig.suptitle(f'{EXP_NAME[exp]} — test confusion matrices (counts; colour = share of the gold row)',
                 x=0.01, ha='left', fontsize=11)
    fig.tight_layout()
    plt.show()

### 2.4 Where the routing errors are

Left: **false records by true class** — which non-transactions slip into extraction. Right: **missed
transactions by sender**. Errors cluster in whole templates (e.g. one wallet's "send" wording), which is why
the intervals above are wide.

In [ ]:
if len(CLS):
    rows = [f"{NAME[r.model]} · {r.experiment[:3]}" for r in CLS.itertuples()]
    fa = [r.meta['test']['routing']['false_accepts_by_class'] for r in CLS.itertuples()]
    fa_cls = sorted({c for d in fa for c in d})
    missed = []
    for r in CLS.itertuples():
        p = PREDS[(r.experiment, r.model)]
        missed.append(p[(p.gold_multiclass == TX) & (p.routed_to_extraction == 0)].sender.value_counts())
    senders = sorted({s for m in missed for s in m.index})
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(4 + 1.0 * len(fa_cls) + 0.75 * len(senders), 0.9 + 0.42 * len(rows)),
                                 gridspec_kw={'width_ratios': [max(len(fa_cls), 1), max(len(senders), 1)]})
    A = np.array([[d.get(c, 0) for c in fa_cls] for d in fa], float)
    heat(a1, A, rows, fa_cls, fmt='{:.0f}', vmin=0, vmax=max(A.max(), 1), cmap='Oranges')
    a1.set_title('false records, by true class')
    B = np.array([[m.get(s, 0) for s in senders] for m in missed], float)
    heat(a2, B, rows, senders, fmt='{:.0f}', vmin=0, vmax=max(B.max(), 1), cmap='Oranges')
    a2.set_yticks([])
    a2.set_title('missed transactions, by sender')
    fig.tight_layout()
    plt.show()

### 2.5 Stage 2 on its own — extraction with a perfect Stage 1

The v1 extractor (retrained on the benchmark's train split) on every **gold** transaction of the test set.
`counterparty` = merchant / recipient / sender, scored with precision/recall because it can legitimately be
empty. `direction` (debit/credit) is derived from the predicted type. `reference` has no gold label and is
not scored.

In [ ]:
xp = os.path.join(RES, 'extraction', 'metrics.json')
EXT = json.load(open(xp, encoding='utf-8')) if os.path.exists(xp) else None
if EXT:
    acc = EXT['fields']['accuracy']
    order = sorted(acc, key=acc.get, reverse=True)
    h = EXT['headline']
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 3.4), gridspec_kw={'width_ratios': [3, 1.2]})
    bars(a1, order, [acc[f] for f in order], V1_COLOR)
    style(a1, f"field accuracy on {EXT['n_test_gold_transactions']} gold transactions", 'accuracy')
    a1.tick_params(axis='x', rotation=30)
    bars(a2, ['full record', 'core record', 'counterparty\nF1'],
         [h['full_record_exact'], h['core_record_exact'], h['counterparty_f1']], V1_COLOR)
    style(a2, 'whole-record exact match')
    fig.tight_layout()
    plt.show()
    display(pd.DataFrame(EXT['fields']['prf']).T[['precision', 'recall', 'f1', 'tp', 'fp', 'fn']])

    bs = pd.DataFrame(EXT['by_sender']).T
    keep = ['amount_acc', 'currency_acc', 'counterparty_f1', 'date_acc', 'direction_acc', 'account_last4_acc',
            'full_record_exact']
    fig, ax = plt.subplots(figsize=(9, 0.8 + 0.42 * len(bs)))
    heat(ax, bs[keep].astype(float).values, list(bs.index), [k.replace('_acc', '') for k in keep])
    ax.set_title('extraction by sender (test gold transactions)')
    plt.show()
else:
    print('no extraction results yet:', xp)

### 2.6 End to end — Stage 1 errors included

A record is **correct** only if the message is a gold transaction, Stage 1 routes it, and every field is right.
Missed transactions and false records (non-transactions turned into records) are counted against the
pipeline. The **oracle** row uses the gold labels as Stage 1: the best any classifier could reach with this
extractor.

In [ ]:
E2E = []
for d in sorted(glob.glob(os.path.join(RES, 'end_to_end', '*', '*'))):
    if os.path.exists(os.path.join(d, 'DONE.json')):
        m = json.load(open(os.path.join(d, 'metrics.json'), encoding='utf-8'))
        E2E.append({'experiment': m['experiment'], 'model': m['model'], 'records_produced': m['records_produced'],
                    'correct_records': m['correct_records'], 'missed_transactions': m['missed_transactions'],
                    'false_records': m['false_records'], 'exact_recall': m['e2e_full_exact_recall'],
                    'exact_precision': m['e2e_full_exact_precision'], 'exact_f1': m['e2e_full_exact_f1'],
                    **{k: m['headline'].get(k) for k in ('amount_acc', 'counterparty_f1', 'date_acc', 'direction_acc')}})
E2E = pd.DataFrame(E2E)
if len(E2E):
    E2E['_o'] = E2E.model.map({m: i for i, m in enumerate(MODELS + ['oracle'])})
    E2E = E2E.sort_values(['experiment', '_o']).drop(columns='_o')
    display(E2E.set_index(['experiment', 'model']).style.format('{:.3f}', subset=['exact_recall', 'exact_precision',
            'exact_f1', 'amount_acc', 'counterparty_f1', 'date_acc', 'direction_acc']))

    e = E2E[E2E.model != 'oracle']
    oracle = E2E[E2E.model == 'oracle']
    ev = lambda k: (lambda g, s: (lambda q: q[k].iloc[0] if len(q) else None)(e[(e.experiment == inv[g]) & (e.model == s)]))
    eg = [EXP_NAME[x] for x in EXPS]
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.6))
    mx = max(e.missed_transactions.max(), e.false_records.max()) * 1.15
    for ax, (k, title, fmt, yl) in zip(axes, [('missed_transactions', 'missed transactions', '{:.0f}', (0, mx)),
                                              ('false_records', 'false records', '{:.0f}', (0, mx)),
                                              ('exact_f1', 'exact-record F1', '{:.2f}', (0, 1.0))]):
        grouped(ax, eg, RUN_MODELS, ev(k), fmt=fmt, ylim=yl)
        style(ax, title)
    if len(oracle):
        o = oracle.exact_f1.iloc[0]
        axes[2].axhline(o, color=MODEL_COLOR['oracle'], ls='--', lw=1, zorder=1)
        axes[2].text(len(eg) - 0.5, o, f' oracle {o:.2f}', va='bottom', ha='right', fontsize=7, color=TEXT2)
    legend_below(fig, axes[0], ncol=len(RUN_MODELS))
    fig.suptitle('End to end on the test set (lower is better for the first two)', x=0.01, ha='left', fontsize=11)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()
    print('note: exact-record F1 can exceed the oracle — routing away a gold transaction the extractor would get '
          'wrong anyway lowers recall less than it raises precision.')
else:
    print('no end-to-end results yet')

### 2.7 Cost: training time, inference speed, size

Inference is measured on the test set: `batch` = whole test set at once, per SMS; `single` = one message at a
time (median of 50), closer to an app receiving SMS one by one. Transformers on GPU, TF-IDF on CPU — the device
column says which.

In [ ]:
if len(CLS):
    c = ['device', 'train_s', 'ms_per_sms_batch', 'ms_per_sms_single', 'size_mb', 'params_m', 'gpu_train_mb']
    display(CLS.set_index(['experiment', 'model'])[c].style.format(
        {'train_s': '{:.1f}', 'ms_per_sms_batch': '{:.3f}', 'ms_per_sms_single': '{:.2f}', 'size_mb': '{:.1f}',
         'params_m': lambda v: '–' if v is None or pd.isna(v) else f'{v:.0f} M',
         'gpu_train_mb': lambda v: '–' if v is None or pd.isna(v) else f'{v:.0f}'}))
    fig, axes = plt.subplots(1, 3, figsize=(17, 3.6))
    for ax, (k, title, fmt) in zip(axes, [('train_s', 'training time (s, log)', '{:.1f}'),
                                          ('ms_per_sms_single', 'ms per SMS, one at a time (log)', '{:.2f}'),
                                          ('size_mb', 'model size on disk (MB, log)', '{:.1f}')]):
        grouped(ax, eg, RUN_MODELS, lambda g, s, k=k: val(k)(inv[g], s), fmt=fmt, ylim=None)
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
        ax.yaxis.set_minor_formatter(NullFormatter())
        style(ax, title)
    legend_below(fig, axes[0], ncol=len(RUN_MODELS))
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()

### 2.8 Transformer training curves (validation, per epoch)

In [ ]:
hist = [(r.experiment, r.model, r.meta['train'].get('history') or []) for r in CLS.itertuples()
        if r.family == 'transformer'] if len(CLS) else []
if hist:
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), sharey=True)
    for ax, exp in zip(axes, ('binary', 'multiclass')):
        for e, m, h in hist:
            if e == exp and h:
                ep = [x['epoch'] for x in h]
                ax.plot(ep, [x['val_macro_f1'] for x in h], marker='o', ms=4, lw=2, color=MODEL_COLOR[m], label=NAME[m])
                best = max(h, key=lambda x: x['val_macro_f1'])
                ax.annotate(f"{best['val_macro_f1']:.3f}", (best['epoch'], best['val_macro_f1']), fontsize=7,
                            color=TEXT2, xytext=(0, 5), textcoords='offset points', ha='center')
        style(ax, f'{EXP_NAME[exp]} — validation macro F1', 'macro F1')
        ax.set_xlabel('epoch')
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print('no transformer results yet — they appear here after colab.ipynb section 5 (point BENCH_OUT at the Drive outputs)')

### 2.9 Experiment A (binary) vs B (6-class), same model

Does teaching the classifier *why* a message is not a transaction (OTP, promo, failed, balance …) change what
reaches extraction? Δ = B − A.

In [ ]:
if set(EXPS) == {'binary', 'multiclass'}:
    rows = []
    for m in RUN_MODELS:
        if ('binary', m) in have and ('multiclass', m) in have:
            a = CLS[(CLS.experiment == 'binary') & (CLS.model == m)].iloc[0]
            b = CLS[(CLS.experiment == 'multiclass') & (CLS.model == m)].iloc[0]
            ea = E2E[(E2E.experiment == 'binary') & (E2E.model == m)]
            eb = E2E[(E2E.experiment == 'multiclass') & (E2E.model == m)]
            row = {'model': NAME[m]}
            for k in ('tx_recall', 'tx_precision', 'missed_tx', 'false_tx'):
                row[f'A {k}'], row[f'B {k}'], row[f'Δ {k}'] = a[k], b[k], b[k] - a[k]
            if len(ea) and len(eb):
                row['A exact_f1'], row['B exact_f1'] = ea.exact_f1.iloc[0], eb.exact_f1.iloc[0]
                row['Δ exact_f1'] = row['B exact_f1'] - row['A exact_f1']
            rows.append(row)
    AB = pd.DataFrame(rows).set_index('model')
    display(AB.style.format('{:.3f}').format('{:+.0f}', subset=[c for c in AB if c.startswith('Δ') and ('missed' in c or 'false' in c)])
            .format('{:.0f}', subset=[c for c in AB if c[0] in 'AB' and ('missed' in c or 'false' in c)]))
else:
    print('needs both experiments')

---
# Part 3 — Old evaluation vs the benchmark

The two are **not** the same experiment, so read this as context, not a league table:

* v1's `random` split lets messages from the same template appear in training and test — its ~1.00 detection
  scores measure memorised wordings. The benchmark's `template` split forbids that (and near-duplicates).
* v1's `unseen_wording` is the closest match to the benchmark (both hold out whole templates), but its folds
  cover all 2 200 messages, while the benchmark scores one fixed test set of 336.
* In the benchmark pipeline, v1 is used in two roles: as the **v1 gate** classifier (only its transaction
  decision) and as the Stage-2 **extractor** (its extraction with the `none` class switched off). Section 3.1
  runs v1 **as one whole model, exactly as before**, on the benchmark's own test set, and checks the two roles
  add up to it.

### 3.1 v1 run whole — as before — on the benchmark's test set

Exactly what `evaluate.py` does in each fold (`SmsTxnModel().fit(train)`, then `predict(sms)` on every test
message: one model decides and extracts), but with the benchmark's split: fitted on its **train** messages
(1 541), scored on its **test** messages (336, templates never seen in training), with `evaluate.score`, i.e.
v1's own metric definitions. This is the like-for-like "old system" row for the benchmark.

In [ ]:
if 'V1E' not in globals():
    sys.path.insert(0, V1_DIR)
    import evaluate as V1E
if 'data' not in globals():
    data = [json.loads(l) for l in open(os.path.join(V1_DIR, 'sms_dataset_2200.jsonl'), encoding='utf-8') if l.strip()]
split_ids = json.load(open(os.path.join(REPO, 'sms_v2', 'data', 'splits', f'{SPLIT}.json'), encoding='utf-8'))['ids']
by_id = {r['id']: r for r in data}
t0 = time.time()
v1_whole = V1E.SmsTxnModel().fit([by_id[i] for i in split_ids['train']])
V1_WHOLE_PAIRS = [(by_id[i], v1_whole.predict(by_id[i]['sms'])) for i in split_ids['test']]
V1_WHOLE, _ = V1E.score(V1_WHOLE_PAIRS)
print(f'v1 fitted on {len(split_ids["train"])} train messages, scored on {len(V1_WHOLE_PAIRS)} test messages '
      f'({time.time() - t0:.0f} s)')
display(pd.Series(V1_WHOLE, name='v1 whole, benchmark test set').to_frame().T.style.format(
    lambda v: '–' if v is None else (f'{v:.0f}' if v > 1 else f'{v:.3f}')))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 3.3), gridspec_kw={'width_ratios': [1, 3.2]})
M = np.zeros((2, 2), int)
for r, p in V1_WHOLE_PAIRS:
    M[0 if r['expected']['is_transaction'] else 1, 0 if p['is_transaction'] else 1] += 1
confusion(a1, M, ['transaction', 'not'], 'v1 whole — transaction vs not')
F = ['tx_precision', 'tx_recall', 'tx_f1', 'non_tx_rejected', 'full_record_exact'] + V1E.FIELDS
bars(a2, [f.replace('_', '\n') for f in F], [V1_WHOLE[f] for f in F], V1_COLOR)
style(a2, 'v1 whole — detection, full record and field accuracy (benchmark test set)')
a2.tick_params(axis='x', labelsize=7)
fig.tight_layout()
plt.show()

# the two benchmark roles, side by side with the whole model
if len(CLS) and ('binary', 'v1_gate') in have:
    g = CLS[(CLS.experiment == 'binary') & (CLS.model == 'v1_gate')].iloc[0]
    e = E2E[(E2E.experiment == 'binary') & (E2E.model == 'v1_gate')]
    chk = pd.DataFrame({
        'v1 whole (this cell)': [V1_WHOLE['tx_precision'], V1_WHOLE['tx_recall'], V1_WHOLE['full_record_exact']],
        'benchmark: v1 gate → v1 extractor': [g.tx_precision, g.tx_recall,
                                              e.exact_recall.iloc[0] if len(e) else None]},
        index=['transaction precision', 'transaction recall', 'full record exact / gold transactions'])
    chk['difference'] = chk.iloc[:, 1] - chk.iloc[:, 0]
    display(chk.style.format('{:.4f}'))
    print('0 difference = splitting v1 into gate + extractor changes nothing; the benchmark rows for v1 are '
          'v1 itself.')

In [ ]:
rows = []
if RUN_V1:
    for s in V1_SPLITS:
        r = V1_RES[s]
        rows.append({'evaluation': f'v1 evaluate.py — {s}', 'n test': r['n'], 'transaction F1': r['tx_f1'],
                     'transaction recall': r['tx_recall'], 'non-tx rejected': r['non_tx_rejected'],
                     'full record exact': r['full_record_exact']})
rows.append({'evaluation': 'benchmark template split — v1 whole (as before)', 'n test': V1_WHOLE['n'],
             'transaction F1': V1_WHOLE['tx_f1'], 'transaction recall': V1_WHOLE['tx_recall'],
             'non-tx rejected': V1_WHOLE['non_tx_rejected'], 'full record exact': V1_WHOLE['full_record_exact']})
if len(CLS) and ('binary', 'v1_gate') in have:
    g = CLS[(CLS.experiment == 'binary') & (CLS.model == 'v1_gate')].iloc[0]
    e = E2E[(E2E.experiment == 'binary') & (E2E.model == 'v1_gate')]
    rows.append({'evaluation': 'benchmark template split — v1 gate + v1 extractor', 'n test': g['meta']['n_test'],
                 'transaction F1': g.tx_f1, 'transaction recall': g.tx_recall, 'non-tx rejected': g.non_tx_recall,
                 'full record exact': e.exact_recall.iloc[0] if len(e) else None})
if len(CLS):
    best = CLS.sort_values('macro_f1', ascending=False).iloc[0]
    e = E2E[(E2E.experiment == best.experiment) & (E2E.model == best.model)]
    rows.append({'evaluation': f'benchmark template split — best Stage 1 by macro F1 ({NAME[best.model]}, {best.experiment})',
                 'n test': best['meta']['n_test'], 'transaction F1': best.tx_f1, 'transaction recall': best.tx_recall,
                 'non-tx rejected': best.non_tx_recall,
                 'full record exact': e.exact_recall.iloc[0] if len(e) else None})
if rows:
    T = pd.DataFrame(rows).set_index('evaluation')
    display(T.style.format('{:.3f}', subset=[c for c in T if c != 'n test']).format('{:.0f}', subset=['n test']))
    fig, axes = plt.subplots(1, 2, figsize=(14, 0.9 + 0.5 * len(T)), sharey=True)
    for ax, k in zip(axes, ['transaction F1', 'full record exact']):
        y = np.arange(len(T))[::-1]
        ax.barh(y, T[k].fillna(0), 0.6, color=V1_COLOR, edgecolor='white')
        for yi, v in zip(y, T[k]):
            if pd.notna(v):
                ax.text(v, yi, f' {v:.2f}', va='center', fontsize=7.5, color=TEXT2)
        ax.set_yticks(y, [i.replace(' — ', '\n') for i in T.index], fontsize=7.5)
        ax.set_xlim(0, 1.12)
        style(ax, k, grid='x')
    fig.tight_layout()
    plt.show()